# LeetCode 1182 — Shortest Distance to Target Color bằng Python

Link đề: https://leetcode.com/problems/shortest-distance-to-target-color/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search, Dynamic Programming

> **Bài Premium — đề viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.** Đề trong notebook được viết lại theo bản mô tả công khai (doocs) và chữ ký hàm suy ra từ lời giải Python tham khảo.

Skeleton LeetCode:

```python
class Solution:
    def shortestDistanceColor(self, colors: List[int], queries: List[List[int]]) -> List[int]:
        ...
```

Ghi chú: mỗi truy vấn `[i, c]` hỏi "từ vị trí `i`, ô có màu `c` **gần nhất** cách bao xa?". Mấu chốt của bài là **tiền xử lý (precompute)** một lần để trả lời rất nhiều truy vấn thật nhanh: dùng **hai lượt quét trái → phải và phải → trái** (một dạng DP), hoặc **tìm kiếm nhị phân** trên danh sách vị trí của từng màu.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mảng `colors` chỉ gồm màu `1, 2, 3`; với mỗi truy vấn `(i, c)` trả về khoảng cách `|i - j|` nhỏ nhất sao cho `colors[j] == c`, hoặc `-1` nếu màu `c` không xuất hiện.
2. Nhận ra vì sao cách "mỗi truy vấn quét cả mảng" (`O(n·q)`) quá chậm khi `n, q` tới `5·10^4`.
3. Nắm hai kỹ thuật tiền xử lý: **nearest-left / nearest-right** bằng hai lượt quét, và **bisect** (tìm kiếm nhị phân) trên danh sách chỉ số đã sắp xếp.
4. Biết cách xử lý trường hợp "không có" bằng giá trị lính canh (sentinel) như `-inf` / `inf` thay vì rắc `if` khắp nơi.
5. Nhận dạng pattern "nhiều truy vấn trên mảng tĩnh → tiền xử lý một lần, trả lời `O(1)` hoặc `O(log n)` mỗi truy vấn".

## 2. Đề bài & ví dụ

Cho mảng `colors` mà mỗi phần tử là một trong ba màu `1`, `2`, `3`. Cho thêm danh sách truy vấn `queries`, mỗi truy vấn là cặp `[i, c]`.

Với mỗi truy vấn, hãy trả về **khoảng cách ngắn nhất** giữa chỉ số `i` và một chỉ số `j` bất kỳ có `colors[j] == c` — tức là `min |i - j|`. Nếu trong mảng không có màu `c` thì trả về `-1`. Kết quả là một danh sách, phần tử thứ `t` là đáp án cho truy vấn thứ `t`.

**Ví dụ 1:** `colors = [1,1,2,1,3,2,2,3,3]`, `queries = [[1,3],[2,2],[6,1]]` → `[3,0,3]`
- Màu `3` gần chỉ số `1` nhất nằm ở chỉ số `4` (cách `3`).
- Màu `2` gần chỉ số `2` nhất chính là chỉ số `2` (cách `0`).
- Màu `1` gần chỉ số `6` nhất nằm ở chỉ số `3` (cách `3`).

**Ví dụ 2:** `colors = [1,2]`, `queries = [[0,3]]` → `[-1]` (không có màu `3`).

**Ràng buộc:**
- `1 <= colors.length <= 5·10^4`, `1 <= colors[i] <= 3`
- `1 <= queries.length <= 5·10^4`, `queries[i].length == 2`
- `0 <= queries[i][0] < colors.length`, `1 <= queries[i][1] <= 3`

In [1]:
import bisect
import random
import time
from collections import defaultdict
from typing import List

examples = [
    ([1, 1, 2, 1, 3, 2, 2, 3, 3], [[1, 3], [2, 2], [6, 1]], [3, 0, 3]),
    ([1, 2], [[0, 3]], [-1]),
]
for colors, queries, want in examples:
    print("colors =", colors, "| queries =", queries, "-> mong đợi", want)

colors = [1, 1, 2, 1, 3, 2, 2, 3, 3] | queries = [[1, 3], [2, 2], [6, 1]] -> mong đợi [3, 0, 3]
colors = [1, 2] | queries = [[0, 3]] -> mong đợi [-1]


## 3. Kiến thức nền

### 3.1 Khoảng cách giữa hai chỉ số

Khoảng cách giữa chỉ số `i` và `j` trên mảng một chiều là `abs(i - j)`. Hàm `abs` (absolute value — giá trị tuyệt đối) bỏ dấu âm, nên thứ tự `i`, `j` không quan trọng.

In [2]:
print(abs(1 - 4), abs(4 - 1), abs(6 - 3))
assert abs(1 - 4) == abs(4 - 1) == 3

3 3 3


### 3.2 Gom chỉ số theo màu bằng `defaultdict(list)`

`defaultdict(list)` là một dict mà khi truy cập khóa chưa có, nó tự tạo `[]`. Duyệt mảng **từ trái sang phải** và `append` chỉ số vào danh sách của màu tương ứng → mỗi danh sách **tự động tăng dần** (đã sắp xếp), rất tiện cho tìm kiếm nhị phân.

In [3]:
colors = [1, 1, 2, 1, 3, 2, 2, 3, 3]
pos = defaultdict(list)
for idx, col in enumerate(colors):
    pos[col].append(idx)
print(dict(pos))
assert pos[3] == [4, 7, 8]

{1: [0, 1, 3], 2: [2, 5, 6], 3: [4, 7, 8]}


### 3.3 Module `bisect` — tìm kiếm nhị phân có sẵn

`bisect.bisect_left(a, x)` trả về vị trí **đầu tiên** trong danh sách tăng dần `a` mà `a[p] >= x` (vị trí "chèn x vào mà vẫn giữ thứ tự"). Như vậy:
- `a[p]` (nếu `p < len(a)`) là phần tử **nhỏ nhất ≥ x** → ứng viên gần nhất **bên phải (hoặc trùng)**.
- `a[p-1]` (nếu `p > 0`) là phần tử **lớn nhất < x** → ứng viên gần nhất **bên trái**.

Độ phức tạp `O(log n)`.

In [4]:
a = [4, 7, 8]
for x in [1, 4, 5, 8, 9]:
    p = bisect.bisect_left(a, x)
    left = a[p - 1] if p > 0 else None
    right = a[p] if p < len(a) else None
    print(f"x={x}: p={p}, trái gần nhất={left}, phải gần nhất (>=x)={right}")

x=1: p=0, trái gần nhất=None, phải gần nhất (>=x)=4
x=4: p=0, trái gần nhất=None, phải gần nhất (>=x)=4
x=5: p=1, trái gần nhất=4, phải gần nhất (>=x)=7
x=8: p=2, trái gần nhất=7, phải gần nhất (>=x)=8
x=9: p=3, trái gần nhất=8, phải gần nhất (>=x)=None


### 3.4 Giá trị lính canh (sentinel) `float('inf')`

Khi "chưa thấy màu nào bên trái", ta có thể lưu `-inf` thay cho chỉ số; khi đó `i - (-inf) = inf` — một khoảng cách "vô cùng" sẽ tự thua mọi khoảng cách thật trong phép `min`. Cuối cùng, nếu đáp án vẫn là `inf` thì đổi thành `-1`. Mẹo này giúp code gọn, ít nhánh `if`.

In [5]:
INF = float('inf')
print(min(5 - (-INF), 3), INF > 10**18)
best = min(INF, INF)
print(-1 if best == INF else best)

3 True
-1


### 3.5 Mảng 2 chiều nhỏ `[[...] for _ in range(4)]`

Ta sẽ lưu `left[c][i]` = chỉ số gần nhất `≤ i` có màu `c`. Vì màu đánh số `1..3`, tạo 4 hàng (bỏ hàng `0`) để chỉ số màu dùng trực tiếp. **Chú ý:** không viết `[[x]*n]*4` vì 4 hàng sẽ là **cùng một list** (lỗi aliasing).

In [6]:
n = 5
good = [[-1] * n for _ in range(4)]
bad = [[-1] * n] * 4
good[1][0] = 9
bad[1][0] = 9
print("good:", good)
print("bad :", bad)
assert bad[2][0] == 9  # bị dính sang hàng khác!

good: [[-1, -1, -1, -1, -1], [9, -1, -1, -1, -1], [-1, -1, -1, -1, -1], [-1, -1, -1, -1, -1]]
bad : [[9, -1, -1, -1, -1], [9, -1, -1, -1, -1], [9, -1, -1, -1, -1], [9, -1, -1, -1, -1]]


## 4. Ý tưởng nền tảng / trực giác

Tính tay truy vấn `(1, 3)` trên `colors = [1,1,2,1,3,2,2,3,3]`:

| j | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
|---|---|---|---|---|---|---|---|---|---|
| colors[j] | 1 | 1 | 2 | 1 | **3** | 2 | 2 | **3** | **3** |
| \|1 - j\| | 1 | 0 | 1 | 2 | **3** | 4 | 5 | 6 | 7 |

Các ô màu `3` có khoảng cách `3, 6, 7` → nhỏ nhất `3`. Quan sát quan trọng:

- Ô gần nhất **chỉ có thể** là ô màu `c` gần nhất **bên trái** (hoặc tại) `i`, hoặc ô màu `c` gần nhất **bên phải** `i`. Các ô xa hơn về cùng một phía chắc chắn xa hơn.
- Vậy với mỗi `(i, c)` chỉ cần biết **2 số**: `L = ` chỉ số lớn nhất `≤ i` có màu `c`, và `R = ` chỉ số nhỏ nhất `≥ i` có màu `c`. Đáp án là `min(i - L, R - i)`.

In [7]:
colors = [1, 1, 2, 1, 3, 2, 2, 3, 3]
i, c = 1, 3
dists = [(j, abs(i - j)) for j in range(len(colors)) if colors[j] == c]
print("các ô màu", c, "và khoảng cách:", dists)
L = max([j for j in range(i + 1) if colors[j] == c], default=None)
R = min([j for j in range(i, len(colors)) if colors[j] == c], default=None)
print("L =", L, "R =", R)

các ô màu 3 và khoảng cách: [(4, 3), (7, 6), (8, 7)]
L = None R = 4


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** với mỗi truy vấn, duyệt toàn bộ mảng, lấy `min |i - j|` trên những `j` có màu `c`. Không thấy thì `-1`.

Đây là cách "dịch thẳng đề ra code" — rất đáng tin để làm **chuẩn đối chiếu** (oracle) khi test các cách nhanh hơn.

In [8]:
def brute(colors: List[int], queries: List[List[int]]) -> List[int]:
    res = []
    for i, c in queries:
        best = -1
        for j, col in enumerate(colors):
            if col == c and (best == -1 or abs(i - j) < best):
                best = abs(i - j)
        res.append(best)
    return res


for colors, queries, want in examples:
    got = brute(colors, queries)
    print(got, want)
    assert got == want

[3, 0, 3] [3, 0, 3]
[-1] [-1]


**Biến thể brute force "loang ra hai phía":** từ `i` thử `d = 0, 1, 2, ...`, kiểm tra `i - d` và `i + d`. Dừng ngay khi gặp màu `c`. Trung bình nhanh hơn nhiều, nhưng **worst case** (màu không tồn tại hoặc ở rất xa) vẫn `O(n)` mỗi truy vấn.

In [9]:
def brute_expand(colors, queries):
    n = len(colors)
    res = []
    for i, c in queries:
        ans = -1
        for d in range(n):
            if (i - d >= 0 and colors[i - d] == c) or (i + d < n and colors[i + d] == c):
                ans = d
                break
        res.append(ans)
    return res


for colors, queries, want in examples:
    assert brute_expand(colors, queries) == want
print(brute_expand(*examples[0][:2]))

[3, 0, 3]


**Độ phức tạp:** `O(n · q)` thời gian, `O(1)` bộ nhớ phụ (không kể output). Với `n = q = 5·10^4` là `2.5·10^9` phép so sánh → **TLE** chắc chắn.

**Nút thắt:** mỗi truy vấn làm lại cùng một công việc "tìm ô màu `c` gần `i`" — mảng `colors` **không đổi** giữa các truy vấn, nên ta lãng phí. Đo thử với kích thước nhỏ để cảm nhận tốc độ tăng theo `n · q`:

In [10]:
random.seed(0)
for n in [500, 1000, 2000]:
    cols = [random.choice([1, 2]) for _ in range(n)]   # không có màu 3 -> worst case
    qs = [[random.randrange(n), 3] for _ in range(n)]
    t = time.perf_counter(); brute(cols, qs); dt = time.perf_counter() - t
    print(f"n=q={n}: {dt:.3f}s")

n=q=500: 0.005s
n=q=1000: 0.023s
n=q=2000: 0.091s


## 6. Bước đột phá

**Insight:** mảng tĩnh + rất nhiều truy vấn → **tiền xử lý một lần**. Từ mục 4 ta biết chỉ cần hai số `L(i, c)` và `R(i, c)`.

**Quy tắc DP cho `L`** (quét trái → phải):
- Nếu `colors[i] == c` thì `L(i, c) = i`.
- Ngược lại `L(i, c) = L(i - 1, c)` (ô gần nhất bên trái của `i` cũng chính là của `i - 1`).

**Quy tắc DP cho `R`** (quét phải → trái), đối xứng:
- Nếu `colors[i] == c` thì `R(i, c) = i`, ngược lại `R(i, c) = R(i + 1, c)`.

**Invariant:** sau khi xử lý xong vị trí `i` trong lượt trái → phải, `last[c]` đúng bằng chỉ số lớn nhất `≤ i` có màu `c` (hoặc `-inf` nếu chưa có).

Chỉ có 3 màu nên bảng `3 × n` là đủ: tiền xử lý `O(3n)`, mỗi truy vấn `O(1)`.

## 7. Các cách tiếp theo

### Cách 2 — Tìm kiếm nhị phân trên danh sách vị trí của từng màu

Gom `pos[c]` = danh sách chỉ số có màu `c` (tự tăng dần). Với truy vấn `(i, c)`: `p = bisect_left(pos[c], i)`; ứng viên là `pos[c][p]` (bên phải hoặc trùng) và `pos[c][p-1]` (bên trái). Cách này **tổng quát** cho số màu lớn tùy ý (vì bộ nhớ chỉ `O(n)` chứ không phải `O(n · số_màu)`).

In [11]:
def solve_bisect(colors: List[int], queries: List[List[int]]) -> List[int]:
    pos = defaultdict(list)
    for idx, col in enumerate(colors):
        pos[col].append(idx)
    res = []
    for i, c in queries:
        a = pos.get(c)
        if not a:
            res.append(-1)
            continue
        p = bisect.bisect_left(a, i)
        best = float('inf')
        if p < len(a):
            best = a[p] - i
        if p > 0:
            best = min(best, i - a[p - 1])
        res.append(best)
    return res


for colors, queries, want in examples:
    assert solve_bisect(colors, queries) == want
print(solve_bisect(*examples[0][:2]))

[3, 0, 3]


**Độ phức tạp:** tiền xử lý `O(n)`, mỗi truy vấn `O(log n)` → tổng `O(n + q log n)`; bộ nhớ `O(n)`.

### Cách 3 — Hai lượt quét (nearest-left / nearest-right), trả lời `O(1)`

Theo đúng quy tắc DP ở mục 6: `left[c][i]` và `right[c][i]`. Dùng `-inf`/`inf` làm lính canh nên công thức trả lời chỉ là `min(i - left[c][i], right[c][i] - i)`.

In [12]:
def solve_two_pass(colors: List[int], queries: List[List[int]]) -> List[int]:
    n = len(colors)
    INF = float('inf')
    left = [[-INF] * n for _ in range(4)]
    right = [[INF] * n for _ in range(4)]
    last = [-INF] * 4
    for i in range(n):
        last[colors[i]] = i
        for c in (1, 2, 3):
            left[c][i] = last[c]
    last = [INF] * 4
    for i in range(n - 1, -1, -1):
        last[colors[i]] = i
        for c in (1, 2, 3):
            right[c][i] = last[c]
    res = []
    for i, c in queries:
        d = min(i - left[c][i], right[c][i] - i)
        res.append(-1 if d == INF else int(d))
    return res


for colors, queries, want in examples:
    assert solve_two_pass(colors, queries) == want
print(solve_two_pass(*examples[0][:2]))

[3, 0, 3]


**Độ phức tạp:** `O(3n + q)` thời gian, `O(3n)` bộ nhớ. Đây là cách nhanh nhất khi số màu nhỏ (cố định là 3).

### Cách 4 — Bảng khoảng cách trực tiếp `dist[c][i]` (gộp hai lượt)

Thay vì lưu chỉ số, lưu luôn **khoảng cách**: lượt trái → phải `dist[c][i] = 0` nếu `colors[i] == c`, ngược lại `dist[c][i-1] + 1`; lượt phải → trái cập nhật `dist[c][i] = min(dist[c][i], dist[c][i+1] + 1)`. Đây chính là **BFS đa nguồn trên đường thẳng** (multi-source BFS): mọi ô màu `c` là nguồn khoảng cách `0`, lan ra hai phía mỗi bước `+1`.

In [13]:
def solve_dist(colors: List[int], queries: List[List[int]]) -> List[int]:
    n = len(colors)
    INF = float('inf')
    dist = [[INF] * n for _ in range(4)]
    for c in (1, 2, 3):
        d = dist[c]
        for i in range(n):
            if colors[i] == c:
                d[i] = 0
            elif i > 0:
                d[i] = d[i - 1] + 1
        for i in range(n - 2, -1, -1):
            if d[i + 1] + 1 < d[i]:
                d[i] = d[i + 1] + 1
    return [-1 if dist[c][i] == INF else int(dist[c][i]) for i, c in queries]


for colors, queries, want in examples:
    assert solve_dist(colors, queries) == want
print(solve_dist(*examples[0][:2]))

[3, 0, 3]


**Độ phức tạp:** `O(3n + q)` thời gian, `O(3n)` bộ nhớ. Ưu điểm: bảng `dist` dùng lại được cho nhiều bài "khoảng cách tới ký tự/ô gần nhất" (ví dụ LeetCode 821).

### Cách 5 — Tiền xử lý lười theo màu + cache truy vấn trùng

Nếu đề cho số màu rất lớn nhưng truy vấn chỉ chạm vài màu, ta có thể tính `dist` **chỉ khi màu đó được hỏi lần đầu** và lưu lại (memoization). Với bài này (3 màu) không lợi gì đáng kể, nhưng là tư duy hữu ích.

In [14]:
def solve_lazy(colors, queries):
    n = len(colors)
    cache = {}

    def table(c):
        if c not in cache:
            INF = float('inf')
            d = [INF] * n
            for i in range(n):
                if colors[i] == c:
                    d[i] = 0
                elif i > 0:
                    d[i] = d[i - 1] + 1
            for i in range(n - 2, -1, -1):
                d[i] = min(d[i], d[i + 1] + 1)
            cache[c] = d
        return cache[c]

    out = []
    for i, c in queries:
        v = table(c)[i]
        out.append(-1 if v == float('inf') else int(v))
    return out, sorted(cache)


print(solve_lazy(*examples[0][:2]))
print(solve_lazy([1, 2], [[0, 3]]))

([3, 0, 3], [1, 2, 3])
([-1], [3])


## 8. TRACE từng bước

Trace lượt trái → phải và phải → trái của Cách 3 trên `colors = [1,1,2,1,3,2,2,3,3]`, theo dõi `last` (chỉ số gần nhất đã thấy của từng màu). `-` nghĩa là chưa thấy (`-inf`/`inf`).

| i | colors[i] | last trái→phải (1,2,3) | last phải→trái (1,2,3) |
|---|---|---|---|
| 0 | 1 | (0, -, -) | (0, 2, 4) |
| 1 | 1 | (1, -, -) | (1, 2, 4) |
| 4 | 3 | (3, 2, 4) | (-, 5, 4) |
| 6 | 2 | (3, 6, 4) | (-, 6, 7) |

Truy vấn `(1, 3)`: `left[3][1] = -inf`, `right[3][1] = 4` → `min(inf, 3) = 3`. Truy vấn `(6, 1)`: `left[1][6] = 3`, `right[1][6] = inf` → `3`.

In [15]:
def trace_two_pass(colors, queries):
    n = len(colors)
    INF = float('inf')
    fmt = lambda v: '-' if v in (INF, -INF) else str(v)
    left = [[-INF] * n for _ in range(4)]
    right = [[INF] * n for _ in range(4)]
    last = [-INF] * 4
    print("Lượt trái -> phải")
    for i in range(n):
        last[colors[i]] = i
        for c in (1, 2, 3):
            left[c][i] = last[c]
        print(f"  i={i} màu={colors[i]} last=({fmt(last[1])}, {fmt(last[2])}, {fmt(last[3])})")
    last = [INF] * 4
    print("Lượt phải -> trái")
    for i in range(n - 1, -1, -1):
        last[colors[i]] = i
        for c in (1, 2, 3):
            right[c][i] = last[c]
        print(f"  i={i} màu={colors[i]} last=({fmt(last[1])}, {fmt(last[2])}, {fmt(last[3])})")
    for i, c in queries:
        L, R = left[c][i], right[c][i]
        d = min(i - L, R - i)
        print(f"Truy vấn ({i},{c}): L={fmt(L)} R={fmt(R)} -> {(-1 if d == INF else int(d))}")


trace_two_pass([1, 1, 2, 1, 3, 2, 2, 3, 3], [[1, 3], [2, 2], [6, 1]])

Lượt trái -> phải
  i=0 màu=1 last=(0, -, -)
  i=1 màu=1 last=(1, -, -)
  i=2 màu=2 last=(1, 2, -)
  i=3 màu=1 last=(3, 2, -)
  i=4 màu=3 last=(3, 2, 4)
  i=5 màu=2 last=(3, 5, 4)
  i=6 màu=2 last=(3, 6, 4)
  i=7 màu=3 last=(3, 6, 7)
  i=8 màu=3 last=(3, 6, 8)
Lượt phải -> trái
  i=8 màu=3 last=(-, -, 8)
  i=7 màu=3 last=(-, -, 7)
  i=6 màu=2 last=(-, 6, 7)
  i=5 màu=2 last=(-, 5, 7)
  i=4 màu=3 last=(-, 5, 4)
  i=3 màu=1 last=(3, 5, 4)
  i=2 màu=2 last=(3, 2, 4)
  i=1 màu=1 last=(1, 2, 4)
  i=0 màu=1 last=(0, 2, 4)
Truy vấn (1,3): L=- R=4 -> 3
Truy vấn (2,2): L=2 R=2 -> 0
Truy vấn (6,1): L=3 R=- -> 3


In [16]:
def trace_bisect(colors, queries):
    pos = defaultdict(list)
    for idx, col in enumerate(colors):
        pos[col].append(idx)
    print("pos =", dict(pos))
    for i, c in queries:
        a = pos.get(c, [])
        p = bisect.bisect_left(a, i)
        print(f"({i},{c}): a={a}, p={p}, trái={a[p-1] if p > 0 else '-'}, phải={a[p] if p < len(a) else '-'}")


trace_bisect([1, 1, 2, 1, 3, 2, 2, 3, 3], [[1, 3], [2, 2], [6, 1], [0, 2]])

pos = {1: [0, 1, 3], 2: [2, 5, 6], 3: [4, 7, 8]}
(1,3): a=[4, 7, 8], p=0, trái=-, phải=4
(2,2): a=[2, 5, 6], p=0, trái=-, phải=2
(6,1): a=[0, 1, 3], p=3, trái=3, phải=-
(0,2): a=[2, 5, 6], p=0, trái=-, phải=2


## 9. Code nộp LeetCode

Chọn Cách 3 (hai lượt quét, `O(1)` mỗi truy vấn). Dùng số nguyên lính canh `-10**9` / `10**9` thay cho `inf` để tránh lẫn float.

In [17]:
class Solution:
    def shortestDistanceColor(self, colors: List[int], queries: List[List[int]]) -> List[int]:
        n = len(colors)
        BIG = 10**9
        left = [[-BIG] * n for _ in range(4)]
        right = [[BIG] * n for _ in range(4)]
        last = [-BIG] * 4
        for i, col in enumerate(colors):
            last[col] = i
            left[1][i], left[2][i], left[3][i] = last[1], last[2], last[3]
        last = [BIG] * 4
        for i in range(n - 1, -1, -1):
            last[colors[i]] = i
            right[1][i], right[2][i], right[3][i] = last[1], last[2], last[3]
        ans = []
        for i, c in queries:
            d = min(i - left[c][i], right[c][i] - i)
            ans.append(-1 if d >= BIG else d)
        return ans


sol = Solution()
for colors, queries, want in examples:
    got = sol.shortestDistanceColor(colors, queries)
    print(got)
    assert got == want

[3, 0, 3]
[-1]


## 10. Vì sao đúng

1. **Chỉ cần hai ứng viên.** Giả sử `j` là ô màu `c` với `j < L` (xa hơn ô trái gần nhất `L`). Khi đó `i - j > i - L`, nên `j` không bao giờ tốt hơn `L`. Tương tự bên phải với `R`. Vậy đáp án `= min(i - L, R - i)`.
2. **Bảng `left` đúng.** Quy nạp theo `i`: sau bước `i`, `last[c]` là chỉ số lớn nhất `≤ i` có màu `c`. Bước `i` chỉ có thể thay đổi `last[colors[i]]` thành `i` (đúng vì `i` lớn hơn mọi chỉ số trước), các màu khác giữ nguyên (đúng vì `colors[i] ≠ c`).
3. **Bảng `right`** đối xứng.
4. **Trường hợp không có màu:** cả `L = -BIG` và `R = BIG` → khoảng cách `≥ BIG` → trả `-1`. Nếu chỉ một phía có, phía kia cho khoảng cách `≥ BIG - n` rất lớn nên `min` chọn phía thật. (Với `n ≤ 5·10^4`, `BIG = 10^9` an toàn: khoảng cách thật luôn `< BIG`, khoảng cách giả luôn `≥ BIG - n > n`.)

   Cẩn thận: với khoảng cách giả `i + BIG` hoặc `BIG - i`, điều kiện `d >= BIG` chỉ đúng khi **cả hai** phía là giả? Thực ra `BIG - i < BIG` nếu `i > 0`! Nhưng khi màu không tồn tại thì cả hai đều giả, `min(i + BIG, BIG - i) = BIG - i`, có thể `< BIG`. Hãy kiểm tra kỹ ở cell dưới — đây là một cái bẫy tinh vi.

In [18]:
# Kiểm tra cái bẫy: màu không tồn tại, i > 0
print(Solution().shortestDistanceColor([1, 1, 1], [[2, 3], [1, 2]]))

[999999998, 999999999]


Cell trên in ra số rất lớn thay vì `-1` → code ở mục 9 **có lỗi** với sentinel số nguyên! Đây đúng là lý do nhiều người thích `inf` (vì `inf - i = inf`). Cách sửa gọn: so sánh `d > n` (khoảng cách thật không bao giờ vượt quá `n - 1`). Ta viết lại bản nộp chuẩn:

In [19]:
class Solution:
    def shortestDistanceColor(self, colors: List[int], queries: List[List[int]]) -> List[int]:
        n = len(colors)
        BIG = 10**9
        left = [[-BIG] * n for _ in range(4)]
        right = [[BIG] * n for _ in range(4)]
        last = [-BIG] * 4
        for i, col in enumerate(colors):
            last[col] = i
            left[1][i], left[2][i], left[3][i] = last[1], last[2], last[3]
        last = [BIG] * 4
        for i in range(n - 1, -1, -1):
            last[colors[i]] = i
            right[1][i], right[2][i], right[3][i] = last[1], last[2], last[3]
        ans = []
        for i, c in queries:
            d = min(i - left[c][i], right[c][i] - i)
            ans.append(-1 if d >= n else d)   # khoảng cách thật luôn <= n - 1
        return ans


sol = Solution()
print(sol.shortestDistanceColor([1, 1, 1], [[2, 3], [1, 2], [0, 1]]))
assert sol.shortestDistanceColor([1, 1, 1], [[2, 3], [1, 2], [0, 1]]) == [-1, -1, 0]
for colors, queries, want in examples:
    assert sol.shortestDistanceColor(colors, queries) == want

[-1, -1, 0]


**Bài học về chứng minh:** đừng chỉ "cảm thấy" sentinel an toàn — hãy viết ra biểu thức khoảng cách giả nhỏ nhất có thể (`BIG - (n-1)`) và so với khoảng cách thật lớn nhất (`n - 1`). Ngưỡng `n` nằm giữa hai giá trị đó nên phân biệt được tuyệt đối.

## 11. Time / Space Complexity

Gọi `n = len(colors)`, `q = len(queries)`, `K` = số màu (ở đây `K = 3`).

| Cách | Tiền xử lý | Mỗi truy vấn | Tổng thời gian | Bộ nhớ phụ |
|---|---|---|---|---|
| 1. Brute force | 0 | `O(n)` | `O(n·q)` | `O(1)` |
| 2. Bisect | `O(n)` | `O(log n)` | `O(n + q log n)` | `O(n)` |
| 3. Hai lượt quét | `O(K·n)` | `O(1)` | `O(K·n + q)` | `O(K·n)` |
| 4. Bảng `dist` | `O(K·n)` | `O(1)` | `O(K·n + q)` | `O(K·n)` |

Với `K = 3`, Cách 3/4 là tuyến tính. Khi `K` lớn (ví dụ màu đến `10^9`), Cách 2 thắng vì bộ nhớ không phụ thuộc `K`.

## 12. Bộ test

Gồm: ví dụ đề, edge case (1 phần tử, màu không tồn tại, truy vấn tại chính ô màu đó, màu chỉ ở đầu/cuối), đối chiếu ngẫu nhiên với brute force (seed cố định), và test giới hạn `n = q = 5·10^4`.

In [20]:
solvers = {
    "bisect": solve_bisect,
    "two_pass": solve_two_pass,
    "dist": solve_dist,
    "submit": Solution().shortestDistanceColor,
}
edge = [
    ([2], [[0, 2], [0, 1], [0, 3]], [0, -1, -1]),
    ([1, 2, 3], [[0, 3], [2, 1], [1, 2]], [2, 2, 0]),
    ([3, 1, 1, 1, 1], [[4, 3]], [4]),
    ([1, 1, 1, 1, 2], [[0, 2]], [4]),
    ([1, 1, 1], [[2, 3], [1, 2]], [-1, -1]),
]
for colors, queries, want in examples + edge:
    for name, f in solvers.items():
        got = f(colors, queries)
        assert got == want, (name, colors, queries, got, want)
print("ví dụ + edge: OK")

ví dụ + edge: OK


In [21]:
random.seed(1182)
for trial in range(400):
    n = random.randint(1, 30)
    palette = random.sample([1, 2, 3], random.randint(1, 3))
    colors = [random.choice(palette) for _ in range(n)]
    queries = [[random.randrange(n), random.randint(1, 3)] for _ in range(random.randint(1, 20))]
    want = brute(colors, queries)
    assert brute_expand(colors, queries) == want
    for name, f in solvers.items():
        assert f(colors, queries) == want, (name, colors, queries)
print("400 test ngẫu nhiên khớp brute force")

400 test ngẫu nhiên khớp brute force


In [22]:
random.seed(7)
n = q = 5 * 10**4
colors = [random.randint(1, 3) for _ in range(n)]
queries = [[random.randrange(n), random.randint(1, 3)] for _ in range(q)]
for name, f in solvers.items():
    t = time.perf_counter(); out = f(colors, queries); dt = time.perf_counter() - t
    print(f"{name:9s}: {dt:.3f}s")
assert solve_bisect(colors, queries) == solve_two_pass(colors, queries)

colors2 = [1] * n   # màu 2, 3 không tồn tại
queries2 = [[random.randrange(n), random.randint(1, 3)] for _ in range(q)]
assert Solution().shortestDistanceColor(colors2, queries2) == solve_bisect(colors2, queries2)
print("test giới hạn OK")

bisect   : 0.030s


two_pass : 0.031s


dist     : 0.022s


submit   : 0.028s


test giới hạn OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ tìm về một phía

Nhiều bạn chỉ quét sang phải từ `i` (hoặc chỉ dùng `bisect` lấy phần tử `≥ i`), quên mất ô màu `c` có thể nằm bên trái gần hơn.

In [23]:
def wrong_one_side(colors, queries):
    res = []
    for i, c in queries:
        ans = -1
        for j in range(i, len(colors)):
            if colors[j] == c:
                ans = j - i
                break
        res.append(ans)
    return res


print(wrong_one_side([1, 1, 2, 1, 3, 2, 2, 3, 3], [[6, 1]]), "nhưng đúng là", brute([1, 1, 2, 1, 3, 2, 2, 3, 3], [[6, 1]]))

[-1] nhưng đúng là [3]


**Vì sao sai:** màu `1` không có bên phải chỉ số `6` nhưng có ở chỉ số `3` bên trái. **Sửa:** luôn xét cả hai phía (`a[p]` và `a[p-1]`, hoặc `left` và `right`).

### Lỗi 2 — Dùng `bisect` nhưng quên kiểm tra biên `p == 0` / `p == len(a)`

In [24]:
def wrong_bounds(colors, queries):
    pos = defaultdict(list)
    for idx, col in enumerate(colors):
        pos[col].append(idx)
    res = []
    for i, c in queries:
        a = pos[c]
        p = bisect.bisect_left(a, i)
        try:
            res.append(min(a[p] - i, i - a[p - 1]))
        except IndexError:
            res.append("IndexError")
    return res


print(wrong_bounds([1, 2, 1], [[2, 2]]), "đúng là", brute([1, 2, 1], [[2, 2]]))
print(wrong_bounds([2, 1, 2], [[0, 2]]), "đúng là", brute([2, 1, 2], [[0, 2]]))

['IndexError'] đúng là [1]
[-2] đúng là [0]


**Vì sao sai:** khi `p == len(a)` thì `a[p]` lỗi; còn khi `p == 0` thì `a[p-1] = a[-1]` **không lỗi** mà lấy phần tử **cuối** — cho kết quả âm hoặc sai một cách âm thầm (trường hợp thứ hai cho `-2`!). **Sửa:** kiểm tra `if p < len(a)` và `if p > 0` riêng như `solve_bisect`.

### Lỗi 3 — Sentinel số nguyên + điều kiện `-1` sai
Như đã phát hiện ở mục 10: dùng `BIG` rồi kiểm tra `d >= BIG` sẽ hỏng khi màu không tồn tại và `i > 0`.

In [25]:
def wrong_sentinel(colors, queries):
    n = len(colors); BIG = 10**9
    last = [-BIG] * 4; left = []
    for i, col in enumerate(colors):
        last[col] = i; left.append(last[:])
    last = [BIG] * 4; right = [None] * n
    for i in range(n - 1, -1, -1):
        last[colors[i]] = i; right[i] = last[:]
    res = []
    for i, c in queries:
        d = min(i - left[i][c], right[i][c] - i)
        res.append(-1 if d >= BIG else d)
    return res


print(wrong_sentinel([1, 1, 1], [[2, 3]]), "đúng là", brute([1, 1, 1], [[2, 3]]))

[999999998] đúng là [-1]


**Sửa:** dùng `float('inf')`, hoặc so sánh với ngưỡng `n` (khoảng cách thật `≤ n - 1`).

### Lỗi 4 — Tạo bảng 2 chiều bằng phép nhân list
`[[x] * n] * 4` tạo 4 tham chiếu tới **cùng** một list → cập nhật màu `1` làm hỏng luôn màu `2`, `3`.

In [26]:
def wrong_alias(colors, queries):
    n = len(colors); INF = float('inf')
    left = [[-INF] * n] * 4          # SAI: 4 hàng dùng chung
    last = [-INF] * 4
    for i in range(n):
        last[colors[i]] = i
        for c in (1, 2, 3):
            left[c][i] = last[c]
    return [i - left[c][i] for i, c in queries]


print(wrong_alias([1, 2, 3], [[2, 1]]), "(chỉ xét bên trái) đúng phải là", [2])

[0] (chỉ xét bên trái) đúng phải là [2]


**Vì sao sai:** sau vòng `for c`, `left[1][i]` bị ghi đè bởi giá trị của màu `3`. **Sửa:** `[[-INF] * n for _ in range(4)]`.

## 14. Mẹo & nhận dạng pattern

- **"Nhiều truy vấn, dữ liệu không đổi"** → nghĩ ngay tới tiền xử lý: prefix sum, nearest-left/right, sparse table, bisect trên danh sách vị trí.
- **"Gần nhất bên trái/phải"** → hai lượt quét (DP một chiều) hoặc stack đơn điệu (nếu là "gần nhất lớn hơn").
- **Số loại giá trị nhỏ (3 màu, 26 chữ cái)** → bảng `K × n` là chấp nhận được; số loại lớn → dict của danh sách vị trí + bisect.
- **Khoảng cách tới tập điểm trên đường thẳng** = BFS đa nguồn rút gọn thành hai lượt quét.
- Luôn tự hỏi: sentinel của mình có thể "rò rỉ" vào đáp án không?

## 15. Biến thể

1. **LeetCode 821 — Shortest Distance to a Character:** một ký tự duy nhất, trả mảng khoảng cách cho mọi vị trí → đúng Cách 4.
2. **Số màu lớn tùy ý (`1..10^9`)** → Cách 2 (dict + bisect).
3. **Mảng có cập nhật (đổi màu ô)** → giữ `SortedList`/cây cân bằng cho từng màu, mỗi thao tác `O(log n)`.
4. **Mảng vòng (circular)** → khoảng cách `min(|i-j|, n-|i-j|)`; có thể nhân đôi mảng rồi làm như thường.
5. **Lưới 2D, khoảng cách Manhattan tới ô màu `c` gần nhất** → BFS đa nguồn trên lưới (LeetCode 542 — 01 Matrix).

In [27]:
# Minh họa biến thể 3: số màu lớn -> Cách 2 vẫn chạy tốt, bảng K x n thì không
colors = [10**9, 5, 10**9, 7]
print(solve_bisect(colors, [[1, 10**9], [3, 5], [0, 42]]))

[1, 2, -1]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | Quét toàn mảng mỗi truy vấn | `O(n·q)` | `O(1)` | Làm oracle khi test, input nhỏ |
| 2. Bisect | Danh sách vị trí mỗi màu + tìm nhị phân | `O(n + q log n)` | `O(n)` | Số màu lớn / không biết trước |
| 3. Hai lượt quét | Lưu chỉ số gần nhất trái/phải | `O(K·n + q)` | `O(K·n)` | Số màu nhỏ, cần nhanh nhất |
| 4. Bảng `dist` | BFS đa nguồn 1 chiều | `O(K·n + q)` | `O(K·n)` | Muốn trả lời trực tiếp khoảng cách |
| 5. Lười + cache | Chỉ tính màu được hỏi | `O(K'·n + q)` | `O(K'·n)` | Nhiều màu nhưng ít màu được hỏi |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** `colors = [3, 3, 1, 2]`, `queries = [[0, 2], [3, 3], [2, 1]]`. Kết quả là gì?

**Bài 2 (tính tay):** Lập bảng `left[2][i]` và `right[2][i]` cho `colors = [2, 1, 1, 3, 2, 1]`.

**Bài 3 (điền TODO):** hoàn thiện hàm `nearest_left_one_color` trả về mảng chỉ số gần nhất `≤ i` có màu `c` (hoặc `-1`).

**Bài 4 (tự viết lại):** viết lại Cách 2 mà **không** dùng module `bisect` (tự cài tìm kiếm nhị phân).

**Bài 5 (mini challenge):** biến thể vòng tròn: khoảng cách giữa `i`, `j` là `min(|i-j|, n-|i-j|)`.

In [28]:
def nearest_left_one_color(colors, c):
    # TODO: trả về list L với L[i] = chỉ số lớn nhất j <= i có colors[j] == c, hoặc -1
    pass

# Test gợi ý (bỏ comment khi làm xong):
# assert nearest_left_one_color([2, 1, 1, 3, 2, 1], 2) == [0, 0, 0, 0, 4, 4]

In [29]:
def my_bisect_left(a, x):
    # TODO: tự cài tìm kiếm nhị phân, trả về vị trí đầu tiên p với a[p] >= x
    raise NotImplementedError


def circular_distance_color(colors, queries):
    # TODO: biến thể mảng vòng
    raise NotImplementedError

# assert circular_distance_color([1, 2, 2, 2, 2], [[4, 1]]) == [1]

## 18. Gợi ý 3 mức

- **Mức 1:** Ô gần nhất chỉ có thể là ô gần nhất bên trái hoặc bên phải.
- **Mức 2:** Tiền xử lý: với mỗi màu, tính chỉ số gần nhất bên trái của mọi `i` bằng một lượt quét trái → phải; tương tự bên phải.
- **Mức 3:** `last = [-inf]*4`; quét `i` tăng dần, `last[colors[i]] = i`, ghi `left[c][i] = last[c]`; làm ngược lại cho `right`; đáp án `min(i - left, right - i)`, `inf → -1`. Bài 5: nhân đôi/ba mảng hoặc dùng bisect với chỉ số `±n`.

## 19. Đáp án tham khảo

**Bài 1:** `[3, 2, 0]` — màu `2` ở chỉ số `3` (cách `0` là `3`); màu `3` gần chỉ số `3` nhất ở chỉ số `1` (cách `2`); chỉ số `2` chính là màu `1`.

**Bài 2:** `left[2] = [0, 0, 0, 0, 4, 4]`, `right[2] = [0, 4, 4, 4, 4, inf]`.

In [30]:
assert brute([3, 3, 1, 2], [[0, 2], [3, 3], [2, 1]]) == [3, 2, 0]


def nearest_left_one_color(colors, c):
    res, last = [], -1
    for i, col in enumerate(colors):
        if col == c:
            last = i
        res.append(last)
    return res


def my_bisect_left(a, x):
    lo, hi = 0, len(a)
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] < x:
            lo = mid + 1
        else:
            hi = mid
    return lo


def circular_distance_color(colors, queries):
    n = len(colors)
    pos = defaultdict(list)
    for idx, col in enumerate(colors):
        pos[col].append(idx)
    res = []
    for i, c in queries:
        a = pos.get(c)
        if not a:
            res.append(-1); continue
        # ứng viên: hàng xóm trong danh sách và hai đầu (vòng qua)
        p = my_bisect_left(a, i)
        cand = {a[p % len(a)], a[p - 1], a[0], a[-1]}
        res.append(min(min(abs(i - j), n - abs(i - j)) for j in cand))
    return res


assert nearest_left_one_color([2, 1, 1, 3, 2, 1], 2) == [0, 0, 0, 0, 4, 4]
random.seed(3)
for _ in range(300):
    a = sorted(random.sample(range(50), random.randint(0, 10)))
    x = random.randint(-2, 52)
    assert my_bisect_left(a, x) == bisect.bisect_left(a, x)
for _ in range(300):
    n = random.randint(1, 15)
    colors = [random.randint(1, 3) for _ in range(n)]
    qs = [[random.randrange(n), random.randint(1, 3)] for _ in range(5)]
    want = []
    for i, c in qs:
        ds = [min(abs(i - j), n - abs(i - j)) for j in range(n) if colors[j] == c]
        want.append(min(ds) if ds else -1)
    assert circular_distance_color(colors, qs) == want
assert circular_distance_color([1, 2, 2, 2, 2], [[4, 1]]) == [1]
print("Đáp án bài tập: OK")

Đáp án bài tập: OK


## 20. Template nhớ nhanh + Checklist trước khi submit

```python
# nearest-left / nearest-right cho K loại giá trị nhỏ
last = [-INF] * (K + 1)
for i in range(n):
    last[a[i]] = i
    for c in range(1, K + 1): left[c][i] = last[c]
last = [INF] * (K + 1)
for i in range(n - 1, -1, -1):
    last[a[i]] = i
    for c in range(1, K + 1): right[c][i] = last[c]
ans = min(i - left[c][i], right[c][i] - i)   # INF -> -1

# hoặc bisect
p = bisect_left(pos[c], i)  # xét pos[c][p] (nếu p < len) và pos[c][p-1] (nếu p > 0)
```

**Checklist:**
- [ ] Đã xét **cả hai phía** trái và phải?
- [ ] Truy vấn tại chính ô có màu `c` trả `0`?
- [ ] Màu không tồn tại trả `-1` (kiểm tra sentinel không rò rỉ)?
- [ ] `bisect`: đã chặn `p == 0` (tránh `a[-1]`) và `p == len(a)`?
- [ ] Bảng 2 chiều tạo bằng list comprehension, không nhân list?
- [ ] Độ phức tạp `O(n + q)` hoặc `O(n + q log n)`, không `O(n·q)`?